# Assignment 2 — Deep Learning (NNDL)

> Spec PDF: `/Users/tahamajs/Documents/uni/LLM/Deep_UT/This_year/CA2/description/NNDL_Assignment2.pdf`

This notebook contains complete, runnable code and organized report sections for both questions. Fill in the textual answers under each subsection and run the code cells to produce your results.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import math
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms, datasets, models

from sklearn.metrics import confusion_matrix, classification_report

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(42)

device = torch.device('cpu')
if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')

print(f"Using device: {device}")

# Table of Contents
#
- [Question 1 — Rice Leaf Disease Detection with CNN](#question-1--rice-leaf-disease-detection-with-cnn)
  - [1-1. Dataset Preparation (5 pts)](#1-1-dataset-preparation-5-pts)
  - [1-2. Data Preprocessing (20 pts)](#1-2-data-preprocessing-20-pts)
  - [1-3. AlexNet Model](#1-3-alexnet-model)
    - [1-3-1. Implementation (20 pts)](#1-3-1-implementation-20-pts)
    - [1-3-2. Results and Evaluation (15 pts)](#1-3-2-results-and-evaluation-15-pts)
  - [1-4. Paper’s Proposed Model (Transfer Learning)](#1-4-papers-proposed-model-transfer-learning)
    - [1-4-1. Implementation (20 pts)](#1-4-1-implementation-20-pts)
    - [1-4-2. Results and Evaluation (20 pts)](#1-4-2-results-and-evaluation-20-pts)
- [Question 2 — Vehicle Classification in Road Scenes](#question-2--vehicle-classification-in-road-scenes)
  - [2-1. Dataset Preparation (10 pts)](#2-1-dataset-preparation-10-pts)
  - [2-2. Data Preprocessing (15 pts)](#2-2-data-preprocessing-15-pts)
  - [2-3. Implementation (20 pts)](#2-3-implementation-20-pts)
  - [2-4. Evaluation (30 pts)](#2-4-evaluation-30-pts)
  - [2-5. Analysis and Optimization](#2-5-analysis-and-optimization)
    - [2-5-1. Effect of Data Augmentation (10)](#2-5-1-effect-of-data-augmentation-10)
    - [2-5-2. Role of Optimizer/Regularization (10)](#2-5-2-role-of-optimizerregularization-10)
    - [2-5-3. Fine-tuning (10)](#2-5-3-fine-tuning-10)
- [Submission Checklist](#submission-checklist)

# Notebook Overview
- Organized into clear sections per question with matching subsections to the spec (Dataset, Preprocessing, Implementation, Results, Analysis).
- Helpers (training/evaluation utilities) are centralized and referenced by both Q1 and Q2, similar to CA1’s structure.
- Use the checklists and templates to keep reporting concise and complete.

# Question 1 — Rice Leaf Disease Detection with CNN

> Paste the exact question prompt here if needed. This section follows the grading rubric outlined in the PDF.

## 1-1. Dataset Preparation (5 pts)
- Download dataset from Kaggle (Rice Disease Dataset).
- Split into train/val/test with 70/15/15 for 3 classes (Healthy + 3 disease classes).
- Plot class distributions per split (histogram/bar).

## 1-2. Data Preprocessing (20 pts)
- Resize all images to 224×224.
- Normalize pixel values to [0, 1] and apply ImageNet normalization for transfer learning.
- Apply at least 3 suitable augmentations (e.g., RandomResizedCrop, HorizontalFlip, ColorJitter). Justify choices.
- Show a few augmented samples for sanity check.

## 1-3. AlexNet Model

### 1-3-1. Implementation (20 pts)
- Implement AlexNet from scratch (PyTorch).
- Provide a model summary.
- Train for ~30 epochs; choose a reasonable optimizer, LR, weight decay; justify choices.

### 1-3-2. Results and Evaluation (15 pts)
- Plot training/validation loss and accuracy across epochs.
- Compute metrics on the test set: Accuracy, Sensitivity (Recall/TPR), Specificity (TNR).
- Report a confusion matrix and discuss which classes are most/least correctly classified and the common confusions.

## 1-4. Paper’s Proposed Model (Transfer Learning)

### 1-4-1. Implementation (20 pts)
- Implement the proposed architecture using VGG19 pretrained on ImageNet as base (freeze early layers).
- Re-do preprocessing to include ImageNet mean/std normalization for all eval sets.
- Keep number of epochs comparable to AlexNet for fair comparison.

### 1-4-2. Results and Evaluation (20 pts)
- Report training curves and test metrics as above.
- Compare against AlexNet:
  - Did the proposed architecture improve data efficiency (higher accuracy with fewer epochs)?
  - Explain how pretrained weights improved performance.
  - Design choices: why combining VGGNet end layers with Inception-like blocks?
  - Why use Global Pooling instead of fully connected layers at the end?

## Q1 — Overview
- Goal: classify rice leaf images into Healthy + 3 diseases.
- Models: AlexNet (from scratch) and a transfer learning model (VGG19).
- Report: accuracy, per-class sensitivity/specificity, confusion matrix; compare AlexNet vs TL.
- Guidance mirrors CA1 style: data prep → preprocessing → implementation → evaluation → analysis.

### 1-1. Dataset Preparation (5 pts)
- Download Rice Disease Dataset from Kaggle (Healthy + 3 diseases).
- Split: 70% train, 15% val, 15% test (stratified).
- Report class distributions and plot histogram per split.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("anshulm257/rice-disease-dataset")
print('Downloaded to:', path)

RICE_DATA_ROOT = Path('/path/to/rice-disease-dataset')  
RICE_CLASSES = ['Healthy', 'BacterialLeafBlight', 'BrownSpot', 'LeafSmut']  

if not RICE_DATA_ROOT.exists():
    print(f"Dataset not found at {RICE_DATA_ROOT}. Please download and update RICE_DATA_ROOT.")
else:
    print(f"Dataset root: {RICE_DATA_ROOT}")

In [ ]:
from sklearn.model_selection import train_test_split
import shutil

def split_rice_dataset(src_root, dest_root, classes, train_r=0.7, val_r=0.15, test_r=0.15, seed=42):
    dest_root = Path(dest_root)
    for split in ['train', 'val', 'test']:
        for cls in classes:
            (dest_root / split / cls).mkdir(parents=True, exist_ok=True)
    
    class_counts = {'train': {}, 'val': {}, 'test': {}}
    for cls in classes:
        cls_path = Path(src_root) / cls
        if not cls_path.exists():
            print(f"Class {cls} not found at {cls_path}")
            continue
        images = sorted(list(cls_path.glob('*.*')))
        train, temp = train_test_split(images, train_size=train_r, random_state=seed)
        val, test = train_test_split(temp, train_size=val_r/(val_r+test_r), random_state=seed)
        
        for img in train:
            shutil.copy(img, dest_root / 'train' / cls / img.name)
        for img in val:
            shutil.copy(img, dest_root / 'val' / cls / img.name)
        for img in test:
            shutil.copy(img, dest_root / 'test' / cls / img.name)
        
        class_counts['train'][cls] = len(train)
        class_counts['val'][cls] = len(val)
        class_counts['test'][cls] = len(test)
    return class_counts

RICE_SPLIT_ROOT = Path('rice_split')
counts = split_rice_dataset(RICE_DATA_ROOT, RICE_SPLIT_ROOT, RICE_CLASSES)
print('Split counts:', counts)

def plot_split_distribution(counts, classes):
    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    for i, split in enumerate(['train', 'val', 'test']):
        vals = [counts[split].get(c, 0) for c in classes]
        axs[i].bar(classes, vals, color='steelblue')
        axs[i].set_title(f'{split.capitalize()} Distribution')
        axs[i].set_ylabel('Count')
        axs[i].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

plot_split_distribution(counts, RICE_CLASSES)

### 1-2. Data Preprocessing (20 pts)
- Resize to 224×224.
- Normalize pixels [0,1]; for transfer learning, apply ImageNet mean/std.
- Implement ≥3 augmentations (RandomResizedCrop, HorizontalFlip, ColorJitter) and justify choices.
- Show augmented examples.

#### Augmentation Justification (Theory)

**Why these augmentations for rice leaf disease?**

1. **RandomResizedCrop (scale=0.7-1.0)**: 
   - Simulates varying distances from camera to leaf and different leaf sizes.
   - Forces model to recognize diseases at multiple scales (early-stage small lesions vs late-stage large blights).

2. **HorizontalFlip**: 
   - Leaves have no canonical orientation in the field; disease patterns are symmetric.
   - Doubles effective dataset size with zero information loss.

3. **ColorJitter (brightness, contrast, saturation, hue)**:
   - Accounts for varying lighting conditions (morning/noon/evening, cloudy/sunny).
   - Disease symptoms involve color changes (yellowing, browning); slight hue shifts test model's robustness to sensor calibration differences.

**Mathematical Framework**

Augmentation as an expectation over transformation distribution $\mathcal{T}$:
$$
\mathcal{L}_{\text{aug}}(\mathbf{w}) = \mathbb{E}_{T \sim \mathcal{T}} \big[ \mathcal{L}(f_{\mathbf{w}}(T(x)), y) \big]
$$
By training on $T(x)$ instead of $x$, we approximate:
$$
\arg\min_{\mathbf{w}} \mathcal{L}_{\text{aug}}(\mathbf{w}) \approx \arg\min_{\mathbf{w}} \mathbb{E}_{x,y} \big[ \max_{T \in \mathcal{T}} \mathcal{L}(f_{\mathbf{w}}(T(x)), y) \big]
$$
This encourages features invariant to $\mathcal{T}$, improving generalization.

**Inappropriate augmentations to avoid**:
- Vertical flips: leaves are typically photographed top-side up; flipping may confuse venation patterns.
- Aggressive rotation (>30°): unnatural orientations rarely occur in controlled agricultural imaging.
- Cutout/mixup: may remove critical disease markers (small lesions).


In [ ]:
def show_augmented_samples(dataset_path, class_name, transform, n=8):
    from PIL import Image
    
    cls_path = Path(dataset_path) / class_name
    if not cls_path.exists():
        print(f"Class path {cls_path} not found.")
        return
    
    imgs = list(cls_path.glob('*.jpg')) + list(cls_path.glob('*.png')) + list(cls_path.glob('*.jpeg'))
    if not imgs:
        print(f"No images in {cls_path}")
        return
    
    img_path = imgs[0]
    img = Image.open(img_path).convert('RGB')
    
    fig, axs = plt.subplots(2, n//2, figsize=(14, 6))
    axs = axs.ravel()
    
    for i in range(n):
        aug_img = transform(img)
        
        if isinstance(aug_img, torch.Tensor):
            aug_img_np = aug_img.permute(1, 2, 0).numpy()
            
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            
            if aug_img_np.min() < 0:
                aug_img_np = aug_img_np * std + mean
            
            aug_img_np = np.clip(aug_img_np, 0, 1)
        else:
            aug_img_np = np.array(aug_img) / 255.0
        
        axs[i].imshow(aug_img_np)
        axs[i].axis('off')
        axs[i].set_title(f'Aug {i+1}', fontsize=9)
    
    plt.suptitle(f'Augmented samples from: {class_name}', fontsize=12)
    plt.tight_layout()
    plt.show()

if RICE_SPLIT_ROOT.exists():
    train_tfms_q1, _ = get_transforms(224, for_transfer=False)
    show_augmented_samples(RICE_SPLIT_ROOT / 'train', RICE_CLASSES[0], train_tfms_q1, n=8)




### 1-3-1. AlexNet Implementation (20 pts)
- Implement AlexNet from scratch (see cell above for architecture).
- Train for 30 epochs; choose optimizer, LR, weight decay; justify.
- Provide model summary.

#### Training Hyperparameters: Justification

**Optimizer Choice: AdamW**
- Adaptive learning rates per parameter handle varying feature scales (early conv vs late dense layers).
- Decoupled weight decay prevents interaction between gradient adaptation and L2 regularization.

**Learning Rate: $3 \times 10^{-4}$**
- Standard for Adam-family optimizers on vision tasks.
- Lower than SGD ($10^{-2}$ to $10^{-1}$) due to adaptive scaling.

**Weight Decay: $10^{-4}$**
- Prevents overfitting by penalizing large weights:
$$
\mathbf{w}_{t+1} = (1 - \lambda)\mathbf{w}_t - \eta \nabla \mathcal{L}
$$
- Typical range $[10^{-5}, 10^{-3}]$; tuned via validation.

**Batch Size: 32**
- Balances gradient noise (helps escape sharp minima) with memory efficiency.
- Smaller batches (16-32) often generalize better than large batches (128+) on small datasets.

**Epochs: 30**
- Sufficient for convergence on small-to-medium datasets (3k-10k images).
- Use early stopping based on validation loss to prevent overfitting.

**Scheduler: CosineAnnealingLR**
$$
\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\Big(1 + \cos\frac{t\pi}{T}\Big)
$$
- Smoothly decays LR from $\eta_{\max}$ to $\eta_{\min}$ over $T$ epochs.
- Allows initial fast learning, then fine-grained refinement near convergence.


In [ ]:
def build_q1_loaders(split_root, img_size=224, batch_size=32, for_transfer=False):
    train_tfms, eval_tfms = get_transforms(img_size, for_transfer=for_transfer)
    
    train_ds = datasets.ImageFolder(str(split_root / 'train'), transform=train_tfms)
    val_ds = datasets.ImageFolder(str(split_root / 'val'), transform=eval_tfms)
    test_ds = datasets.ImageFolder(str(split_root / 'test'), transform=eval_tfms)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    return train_loader, val_loader, test_loader, train_ds.classes

if RICE_SPLIT_ROOT.exists():
    train_loader_alex, val_loader_alex, test_loader_alex, class_names_alex = build_q1_loaders(
        RICE_SPLIT_ROOT, img_size=224, batch_size=32, for_transfer=False
    )
    print(f"AlexNet classes: {class_names_alex}")
    
    model_alex = AlexNet(num_classes=len(class_names_alex)).to(device)
    print(f"AlexNet parameters: {sum(p.numel() for p in model_alex.parameters()):,}")
    
    history_alex, best_path_alex = fit(
        model_alex, train_loader_alex, val_loader_alex,
        epochs=30, lr=3e-4, weight_decay=1e-4, save_dir='outputs'
    )
    plot_history(history_alex)
    
    # Evaluate
    ckpt_alex = torch.load(best_path_alex, map_location=device)
    model_alex.load_state_dict(ckpt_alex['model_state'])
    results_alex = evaluate_full(model_alex, test_loader_alex, class_names_alex)
    
    print(f"\nAlexNet Final Results:")
    print(f"  Test Accuracy: {results_alex['test_acc']:.4f}")
    print(f"  Per-class Sensitivity: {results_alex['sens']}")
    print(f"  Per-class Specificity: {results_alex['spec']}")




### 1-3-2. AlexNet Results and Evaluation (15 pts)
- Plot training/validation curves (loss + accuracy).
- Report test metrics: Accuracy, per-class Sensitivity, per-class Specificity.
- Show confusion matrix and discuss which classes are most/least correctly classified.

#### Metrics Interpretation Guide

**Accuracy**
$$
\text{Acc} = \frac{\text{TP} + \text{TN}}{\text{Total}}
$$
- Overall correctness; can be misleading for imbalanced classes (e.g., 95% healthy, 5% diseased → trivial 95% accuracy by always predicting healthy).

**Sensitivity (Recall / True Positive Rate)**
$$
\text{Sens}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FN}_c}
$$
- Per-class: "Of all actual class-$c$ samples, what fraction did we detect?"
- Critical for disease detection: high sensitivity = few missed diseases (low false negatives).

**Specificity (True Negative Rate)**
$$
\text{Spec}_c = \frac{\text{TN}_c}{\text{TN}_c + \text{FP}_c}
$$
- Per-class: "Of all non-class-$c$ samples, what fraction did we correctly reject?"
- High specificity = low false alarms (not misclassifying healthy as diseased).

**Confusion Matrix Analysis**
- Diagonal: correct predictions.
- Off-diagonal: common confusions (e.g., BrownSpot ↔ BacterialBlight if visually similar).
- Use to identify which classes need more training data or feature engineering.

**Trade-offs**
- High sensitivity + low specificity: model over-predicts the class (many false positives).
- Low sensitivity + high specificity: model under-predicts (many false negatives).
- Balance via threshold tuning or class weights in loss function.


In [ ]:
ckpt_alex = torch.load(best_path_alex, map_location=device)
model_alex.load_state_dict(ckpt_alex['model_state'])
results_alex = evaluate_full(model_alex, test_loader_q1, class_names_q1)

print('AlexNet evaluation cell ready (uncomment when trained).')

### 1-4-1. Transfer Learning (VGG19) Implementation (20 pts)
- Use VGG19 pretrained on ImageNet; freeze early layers.
- Replace classifier head with new Dense layer for rice classes.
- Re-apply preprocessing with ImageNet mean/std for all splits.
- Train for 30 epochs (same as AlexNet for fair comparison).

In [ ]:
if RICE_SPLIT_ROOT.exists():
    train_loader_vgg, val_loader_vgg, test_loader_vgg, class_names_vgg = build_q1_loaders(
        RICE_SPLIT_ROOT, img_size=224, batch_size=32, for_transfer=True
    )
    print(f"VGG19 classes: {class_names_vgg}")
    
    model_vgg = build_vgg19_tl(num_classes=len(class_names_vgg), freeze_backbone=True)
    trainable = sum(p.numel() for p in model_vgg.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model_vgg.parameters())
    print(f"VGG19 trainable/total params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
    
    history_vgg, best_path_vgg = fit(
        model_vgg, train_loader_vgg, val_loader_vgg,
        epochs=30, lr=3e-4, weight_decay=1e-4, save_dir='outputs'
    )
    plot_history(history_vgg)
    
    ckpt_vgg = torch.load(best_path_vgg, map_location=device)
    model_vgg.load_state_dict(ckpt_vgg['model_state'])
    results_vgg = evaluate_full(model_vgg, test_loader_vgg, class_names_vgg)
    
    print(f"\nVGG19 Transfer Learning Final Results:")
    print(f"  Test Accuracy: {results_vgg['test_acc']:.4f}")
    print(f"  Per-class Sensitivity: {results_vgg['sens']}")
    print(f"  Per-class Specificity: {results_vgg['spec']}")
    
    print("\n" + "="*60)
    print("Q1 MODEL COMPARISON")
    print("="*60)
    print(f"{'Model':<20} | {'Test Acc':<12} | {'Avg Sens':<12} | {'Avg Spec':<12}")
    print("-"*60)
    print(f"{'AlexNet (scratch)':<20} | {results_alex['test_acc']:.4f}       | {results_alex['sens'].mean():.4f}       | {results_alex['spec'].mean():.4f}")
    print(f"{'VGG19 (transfer)':<20} | {results_vgg['test_acc']:.4f}       | {results_vgg['sens'].mean():.4f}       | {results_vgg['spec'].mean():.4f}")
    improvement = (results_vgg['test_acc'] - results_alex['test_acc']) / results_alex['test_acc'] * 100
    print(f"\nImprovement: {improvement:+.2f}%")
    print("="*60)




### 1-4-2. Transfer Learning Results and Evaluation (20 pts)
- Report test metrics as above.
- Compare with AlexNet:
  - Data efficiency (accuracy gain with fewer epochs)?
  - How pretrained weights improved performance.
  - Architectural rationale: VGGNet + Inception/Global Pooling choices.
  - Why Global Pooling instead of FC layers?

#### Theoretical Analysis: Transfer Learning vs From-Scratch

**Data Efficiency**
- Transfer learning achieves higher accuracy with fewer epochs because pretrained weights on ImageNet already encode low-level features (edges, textures) and mid-level patterns (shapes, object parts) that generalize to new tasks.
- From-scratch training (AlexNet) must learn these representations from limited rice-disease data, requiring more epochs and larger datasets to converge.

**How Pretrained Weights Improve Performance**
1. **Feature reuse**: Early convolutional layers learn universal visual features (Gabor-like filters, edge detectors) that are task-agnostic. Freezing these layers prevents overfitting on small datasets.
2. **Faster convergence**: Only the final classifier layers adapt to rice-disease classes, reducing the effective parameter search space.
3. **Regularization effect**: Pretrained weights act as a strong prior, constraining the model to biologically plausible feature hierarchies.

**Architectural Rationale: VGG + Inception + Global Pooling**

*Why VGG19 as backbone?*
- VGG's uniform 3×3 convolutions with deep stacking (16-19 layers) build rich hierarchical features.
- Pretrained on ImageNet (1000 classes, 1.2M images), it captures diverse visual concepts transferable to agricultural imagery.

*Why add Inception-like blocks?*
$$
\text{Inception}(x) = \text{Concat}\Big[\text{Conv}_{1\times1}(x),\, \text{Conv}_{3\times3}(x),\, \text{Conv}_{5\times5}(x),\, \text{MaxPool}(x)\Big]
$$
- Multi-scale receptive fields capture both fine-grained leaf texture (small kernels) and global lesion patterns (large kernels) simultaneously.
- Reduces parameters via $1\times1$ bottlenecks before expensive $3\times3$ and $5\times5$ convolutions.

*Why Global Average Pooling (GAP) instead of Fully Connected layers?*
1. **Parameter reduction**: FC layers (e.g., $7\times7\times512 \to 4096$) introduce ~100M parameters, causing overfitting on small datasets. GAP averages each feature map to a scalar, adding zero parameters.
2. **Spatial invariance**: GAP enforces that class activation can occur anywhere in the feature map, improving robustness to object position/scale.
3. **Interpretation**: Each feature map directly corresponds to a class score, enabling class activation mapping (CAM) for visualization.

**Mathematical Formulation**
- GAP for feature map $f \in \mathbb{R}^{H \times W}$:
$$
\text{GAP}(f) = \frac{1}{HW}\sum_{i=1}^H\sum_{j=1}^W f_{ij}
$$
- Final classifier with $C$ classes and $K$ feature maps:
$$
y_c = \sum_{k=1}^K w_{c,k}\,\text{GAP}(f_k) + b_c,\quad \text{softmax}(\mathbf{y})
$$


In [ ]:
ckpt_vgg = torch.load(best_path_vgg, map_location=device)
model_vgg.load_state_dict(ckpt_vgg['model_state'])
results_vgg = evaluate_full(model_vgg, test_loader_q1_tl, class_names_q1_tl)

print('\nComparison AlexNet vs VGG19 TL:')
print(f"AlexNet  - Test Acc: {results_alex['test_acc']:.4f}")
print(f"VGG19 TL - Test Acc: {results_vgg['test_acc']:.4f}")

print('VGG19 TL evaluation cell ready (uncomment when trained).')

# Question 2 — Vehicle Classification in Road Scenes

> Paste the exact question prompt here if needed. Follow the rubric.

## 2-1. Dataset Preparation (10 pts)
- Prepare dataset and define classes (e.g., car, truck, bus, bike).
- Train/val/test split with clear distribution reporting.

## 2-2. Data Preprocessing (15 pts)
- Resize to 224×224 (or 299×299 for Xception) and normalize.
- Select augmentations suitable for road scenes/remote sensing; justify choices.

## 2-3. Implementation (20 pts)
- Implement baseline CNN or reuse `SimpleCNN` (or AlexNet) with tuned hyperparameters.
- Optionally add transfer learning (e.g., ResNet18/VGG19/Xception) for comparison.

## 2-4. Evaluation (30 pts)
- Plot curves and report metrics on the test set: Accuracy, per-class Recall/Sensitivity, per-class Specificity, confusion matrix.

## 2-5. Analysis and Optimization

### 2-5-1. Effect of Data Augmentation (10)

**Theoretical Background**

Data augmentation artificially expands the training set by applying label-preserving transformations, providing several benefits:

1. **Regularization**: Prevents overfitting by exposing the model to diverse variations of each sample.
2. **Invariance learning**: Encourages the model to learn features robust to translation, rotation, scale, and photometric changes.
3. **Sample efficiency**: Reduces dependence on large labeled datasets—critical for specialized domains like remote sensing.

**Augmentations for Remote Sensing / Road Scenes**

- **Rotation (±45°)**: Satellite/aerial imagery has no canonical orientation; vehicles/buildings appear at arbitrary angles.
- **Horizontal/Vertical Flip**: Geographic scenes lack inherent left-right or up-down bias (unlike faces or text).
- **Random Zoom/Crop**: Simulates different altitudes and sensor resolutions.
- **Color Jitter**: Accounts for varying illumination, atmospheric conditions, and sensor calibration.

**Expected Impact**
$$
\text{Generalization gap} = \mathcal{L}_{\text{val}} - \mathcal{L}_{\text{train}}
$$
Without augmentation, $\mathcal{L}_{\text{train}} \ll \mathcal{L}_{\text{val}}$ (overfitting). Augmentation reduces this gap by increasing effective training set size and feature robustness.

**Experimental Comparison**
- No-Aug: Train without augmentation (only resize/normalize).
- With-Aug: Add rotation, flips, zoom, color jitter.
- Metrics: Compare final test accuracy and validation loss curves.

### 2-5-2. Role of Optimizer/Regularization (10)

**Optimizer Comparison: SGD vs AdamW**

*SGD (Stochastic Gradient Descent)*
$$
\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \nabla_{\mathbf{w}} \mathcal{L}(\mathbf{w}_t)
$$
- Pros: Simple, well-studied; with momentum converges to flatter minima (better generalization).
- Cons: Requires careful LR tuning and warm-up schedules; slower on high-curvature loss surfaces.

*AdamW (Adam with decoupled weight decay)*
$$
\mathbf{m}_t = \beta_1 \mathbf{m}_{t-1} + (1-\beta_1)\mathbf{g}_t,\quad \mathbf{v}_t = \beta_2 \mathbf{v}_{t-1} + (1-\beta_2)\mathbf{g}_t^2
$$
$$
\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \frac{\mathbf{m}_t}{\sqrt{\mathbf{v}_t} + \epsilon} - \lambda \mathbf{w}_t
$$
- Pros: Adaptive per-parameter LR; faster convergence; stable across wide LR ranges.
- Cons: Can overfit without proper weight decay; may converge to sharper minima than SGD+momentum.

*Adagrad (for Q2 MLP classifier)*
$$
\mathbf{w}_{t+1} = \mathbf{w}_t - \frac{\eta}{\sqrt{G_t + \epsilon}}\mathbf{g}_t,\quad G_t = \sum_{\tau=1}^t \mathbf{g}_\tau^2
$$
- Accumulates squared gradients; ideal for sparse features or when different parameters converge at different rates.
- Used in Q2 for the MLP head while Xception features are frozen.

**Regularization Techniques**

1. **Weight Decay (L2 penalty)**:
$$
\mathcal{L}_{\text{reg}} = \mathcal{L}_{\text{CE}} + \frac{\lambda}{2}\|\mathbf{w}\|_2^2
$$
- Penalizes large weights, preventing overfitting.
- Typical values: $\lambda \in [10^{-5}, 10^{-3}]$.

2. **Dropout**:
$$
\tilde{x}_i = \begin{cases} \frac{x_i}{1-p} & \text{with prob. } 1-p \\ 0 & \text{with prob. } p \end{cases}
$$
- Randomly drops units during training, forcing redundant representations.
- Q2 uses dropout=0.4 in the MLP classifier.

3. **Batch Normalization**:
$$
\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}},\quad y = \gamma\hat{x} + \beta
$$
- Stabilizes training by normalizing activations; reduces internal covariate shift.
- Q2 applies BN before sigmoid activation in the enhanced MLP.

**Expected Outcomes**
- SGD+momentum: slower but potentially better generalization.
- AdamW: faster convergence; requires tuning $\lambda$.
- Regularization reduces overfitting (val-train gap); too much hurts expressiveness.

### 2-5-3. Fine-tuning (10)

**Transfer Learning Strategies**

1. **Frozen Feature Extractor** (default):
   - Freeze all backbone layers; train only the classifier head.
   - Pros: Fast, prevents overfitting on small datasets.
   - Cons: Cannot adapt features to domain-specific patterns.

2. **Full Fine-tuning**:
   - Unfreeze all layers; train end-to-end with small LR.
   - Pros: Maximizes performance when data is sufficient.
   - Cons: Risk of catastrophic forgetting and overfitting.

3. **Gradual Unfreezing** (best practice):
   - Stage 1: Train classifier with frozen backbone (5-10 epochs).
   - Stage 2: Unfreeze top $N$ layers (e.g., last conv block); train with LR $\eta/10$.
   - Stage 3: Optionally unfreeze all layers; train with LR $\eta/100$.

**Learning Rate Considerations**

When fine-tuning pretrained models, use discriminative learning rates:
$$
\eta_{\text{backbone}} < \eta_{\text{head}}
$$
- Backbone layers already encode useful features; large LR destroys them.
- Typical ratio: head LR = $10 \times$ backbone LR.

**Mathematical Intuition**

Let $\mathbf{w} = [\mathbf{w}_{\text{backbone}}, \mathbf{w}_{\text{head}}]$. Pretrained weights $\mathbf{w}_{\text{backbone}}^*$ sit near a good local minimum for general vision. Fine-tuning refines this:
$$
\mathbf{w}_{\text{backbone}}^{(t+1)} = \mathbf{w}_{\text{backbone}}^{(t)} - \eta_{\text{small}} \nabla \mathcal{L}
$$
while the head learns task-specific mappings with larger updates.

**Expected Trade-offs**
- **Accuracy**: Fine-tuning improves test accuracy by 1-5% if data >1000 samples/class.
- **Training time**: 2-3× longer than frozen training.
- **Overfitting risk**: Monitor validation loss; stop if divergence occurs.
- **Optimal depth to unfreeze**: Depends on domain shift—larger shift (e.g., medical→natural images) benefits from deeper unfreezing.

**Practical Guidelines**
- Small dataset (<5k images): freeze backbone or unfreeze only last block.
- Medium dataset (5k-50k): gradual unfreezing with LR decay.
- Large dataset (>50k): full fine-tuning with warmup schedule.


#### Theoretical Answers: Q2 Analysis

The following subsections provide theoretical foundations for data augmentation, optimizer/regularization choices, and fine-tuning strategies in vehicle/land-use classification tasks.


## Q2 — Overview
- Goal: classify road-scene vehicles (e.g., car/truck/bus/bike).
- Baseline + optional transfer learning; follow Q1’s flow.
- Evaluate with accuracy, per-class sensitivity/specificity, and confusion matrix; analyze augmentations, optimizer/regularization, and fine-tuning.

### 2-1. Dataset Preparation (10 pts)
- Download UC-Merced Land Use dataset from Kaggle.
- Split 80/20 for train/test.
- Report class distributions and plot histogram.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("abdulhasibuddin/uc-merced-land-use-dataset")
print('Downloaded to:', path)

UC_MERCED_ROOT = Path('/path/to/uc-merced')  
UC_MERCED_CLASSES = [ 
    'agricultural', 'airplane', 'baseballdiamond', 'beach', 'buildings',
    'chaparral', 'denseresidential', 'forest', 'freeway', 'golfcourse',
    'harbor', 'intersection', 'mediumresidential', 'mobilehomepark', 'overpass',
    'parkinglot', 'river', 'runway', 'sparseresidential', 'storagetanks', 'tenniscourt'
]

if not UC_MERCED_ROOT.exists():
    print(f"UC-Merced not found at {UC_MERCED_ROOT}. Please download and update.")
else:
    print(f"UC-Merced root: {UC_MERCED_ROOT}")

In [ ]:
def split_ucmerced(src_root, dest_root, classes, train_r=0.8, seed=42):
    dest_root = Path(dest_root)
    for split in ['train', 'test']:
        for cls in classes:
            (dest_root / split / cls).mkdir(parents=True, exist_ok=True)
    
    counts = {'train': {}, 'test': {}}
    for cls in classes:
        cls_path = Path(src_root) / cls
        if not cls_path.exists():
            continue
        images = sorted(list(cls_path.glob('*.*')))
        train, test = train_test_split(images, train_size=train_r, random_state=seed)
        for img in train:
            shutil.copy(img, dest_root / 'train' / cls / img.name)
        for img in test:
            shutil.copy(img, dest_root / 'test' / cls / img.name)
        counts['train'][cls] = len(train)
        counts['test'][cls] = len(test)
    return counts

UC_MERCED_SPLIT = Path('ucmerced_split')
counts_ucm = split_ucmerced(UC_MERCED_ROOT, UC_MERCED_SPLIT, UC_MERCED_CLASSES)
print('UC-Merced split counts:', counts_ucm)



### 2-2. Data Preprocessing (15 pts)
- Resize to 299×299 (for Xception).
- Normalize with ImageNet mean/std.
- Implement augmentations: rotation, zoom, horizontal/vertical flip.
- Justify choices for remote sensing imagery.
- Show augmented examples.

#### Q2 Augmentation Rationale (Remote Sensing)

**Why 299×299 for Xception?**
- Xception architecture was designed and pretrained on ImageNet at 299×299 resolution.
- Using different sizes requires resampling that may degrade feature alignment with pretrained weights.

**Remote Sensing / Land Use Specific Augmentations**

1. **Rotation (±45°)**: 
   - Satellite/aerial imagery has arbitrary orientation; north is not always "up."
   - Buildings, roads, and vehicles appear at all angles.

2. **Horizontal + Vertical Flip**:
   - Unlike natural images (faces have canonical orientation), land use scenes are rotationally symmetric.
   - Doubles dataset with zero semantic change.

3. **Random Zoom (scale=0.7-1.0)**:
   - Simulates different altitudes and zoom levels of aerial sensors.
   - Forces model to recognize classes at multiple scales (e.g., dense residential vs sparse residential differs in zoom).

4. **Color Jitter**:
   - Atmospheric conditions (haze, clouds), time of day, and sensor calibration cause color variations.
   - Robustness to these is critical for real-world deployment across different sensors/locations.

**Augmentations to AVOID**:
- Elastic deformations: geometric structures (roads, buildings) have rigid shapes; warping violates physical constraints.
- Cutout: may remove critical discriminative regions (e.g., runway markings, storage tank circles).
- Extreme brightness/contrast: satellite imagery is calibrated; extreme shifts unrealistic.

**Theoretical Foundation**

Test-time augmentation (TTA) can further boost accuracy:
$$
p_{\text{TTA}}(y|x) = \frac{1}{|A|}\sum_{T \in A} p(y | T(x))
$$
Average predictions over multiple augmented versions of the test image to reduce variance.


In [ ]:
IMAGENET_MEAN_Q2 = [0.485, 0.456, 0.406]
IMAGENET_STD_Q2 = [0.229, 0.224, 0.225]

def get_transforms_q2(img_size=299):
    train_tfms = transforms.Compose([
        transforms.Resize(int(img_size * 1.15)),
        transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
        transforms.RandomRotation(45),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN_Q2, IMAGENET_STD_Q2)
    ])
    test_tfms = transforms.Compose([
        transforms.Resize(int(img_size * 1.15)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN_Q2, IMAGENET_STD_Q2)
    ])
    return train_tfms, test_tfms

train_tfms_q2, _ = get_transforms_q2()
show_augmented_samples(UC_MERCED_SPLIT / 'train', UC_MERCED_CLASSES[0], train_tfms_q2)



### 2-3. Implementation (20 pts)
- Feature extractor: Xception pretrained on ImageNet (frozen).
- Classifier: Enhanced MLP (Normalization → Sigmoid → Dropout(0.4) → Softmax).
- Optimizer: Adagrad for MLP.
- Train for 50 epochs; report architecture and hyperparameters.

#### Q2 Architecture: Xception + Enhanced MLP

**Why Xception?**

Xception (Extreme Inception) improves upon Inception by replacing standard convolutions with depthwise separable convolutions:
$$
\text{Standard Conv: } y = \sigma(W * x),\quad W \in \mathbb{R}^{C_{\text{out}} \times C_{\text{in}} \times k \times k}
$$
$$
\text{Depthwise Separable: } y = \sigma(W_{\text{point}} \cdot (W_{\text{depth}} * x))
$$
where $W_{\text{depth}} \in \mathbb{R}^{C_{\text{in}} \times 1 \times k \times k}$ (spatial filtering per channel) and $W_{\text{point}} \in \mathbb{R}^{C_{\text{out}} \times C_{\text{in}} \times 1 \times 1}$ (channel mixing).

**Parameter Efficiency**
- Standard conv: $C_{\text{out}} \times C_{\text{in}} \times k^2$ parameters.
- Depthwise separable: $C_{\text{in}} \times k^2 + C_{\text{out}} \times C_{\text{in}}$ parameters.
- Reduction factor: $\approx \frac{1}{C_{\text{out}}} + \frac{1}{k^2}$ (e.g., ~8-9× fewer for $k=3$, $C_{\text{out}}=256$).

**Why freeze Xception backbone?**
- Pretrained on ImageNet (1000 classes, natural images); early-mid layers learn universal features (edges, textures, object parts).
- UC-Merced has only ~2100 images (21 classes × 100 images); full fine-tuning would overfit.
- Freezing acts as regularization and speeds up training.

**Enhanced MLP Classifier Design**

Standard approach: $\text{GAP} \to \text{Dense}(C)$ is simple but may underfit.

Enhanced pipeline:
1. **Global Average Pooling**: reduces $H \times W \times 2048$ feature maps to $2048$-dim vector.
2. **Batch Normalization**: stabilizes activations entering the MLP.
3. **Sigmoid activation**: introduces non-linearity; alternative to ReLU (prevents dead neurons in small MLPs).
4. **Dropout(0.4)**: aggressive regularization to prevent overfitting on limited data.
5. **Dense(21) + Softmax**: final classification layer.

**Why Sigmoid in hidden layer?**
$$
\sigma(z) = \frac{1}{1 + e^{-z}},\quad \sigma'(z) = \sigma(z)(1-\sigma(z))
$$
- Bounded output $[0,1]$ acts as gating; interpretable as "feature importance."
- Smoother gradients than ReLU near zero (no dead-unit problem in small networks).

**Why Adagrad optimizer for MLP?**
$$
\mathbf{w}_{t+1} = \mathbf{w}_t - \frac{\eta}{\sqrt{G_t + \epsilon}} \nabla \mathcal{L},\quad G_t = \sum_{\tau=1}^t \nabla \mathcal{L}_\tau^2
$$
- Adapts LR per parameter based on historical gradient magnitude.
- Ideal when different features (frozen Xception outputs) have vastly different scales.
- Accumulation of $G_t$ ensures frequently-updated parameters (common features) get smaller LR; rare features get larger LR.

**Trade-offs**
- Pro: Parameter-efficient; fast training; strong baseline for land use classification.
- Con: Limited capacity compared to full fine-tuning; may underperform on very fine-grained distinctions.


In [ ]:

try:
    import timm
    HAS_TIMM = True
except:
    HAS_TIMM = False
    print('timm not installed. Install via: pip install timm')

class XceptionMLP(nn.Module):
    def __init__(self, num_classes=21, dropout=0.4):
        super().__init__()
        if HAS_TIMM:
            self.feature_extractor = timm.create_model('xception', pretrained=True, num_classes=0)
            for p in self.feature_extractor.parameters():
                p.requires_grad = False
            feat_dim = self.feature_extractor.num_features
        else:
            base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
            self.feature_extractor = nn.Sequential(*list(base.children())[:-1])
            for p in self.feature_extractor.parameters():
                p.requires_grad = False
            feat_dim = 2048
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(feat_dim),
            nn.Sigmoid(),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, num_classes)
        )
    
    def forward(self, x):
        feats = self.feature_extractor(x)
        if feats.dim() == 4:
            feats = self.global_avg_pool(feats)
        feats = feats.view(feats.size(0), -1)
        out = self.mlp(feats)
        return out

model_q2 = XceptionMLP(num_classes=len(UC_MERCED_CLASSES)).to(device)
print(model_q2)



In [ ]:
if UC_MERCED_SPLIT.exists():
    train_loader_q2, val_loader_q2, test_loader_q2, class_names_q2 = build_q2_loaders(
        UC_MERCED_SPLIT, img_size=299, batch_size=32
    )
    print(f"Q2 classes: {class_names_q2}")
    print(f"Train batches: {len(train_loader_q2)}, Val: {len(val_loader_q2)}, Test: {len(test_loader_q2)}")
    model_q2 = XceptionMLP(num_classes=len(class_names_q2)).to(device)
    history_q2, best_path_q2 = fit_with_optimizer(
        model_q2, train_loader_q2, val_loader_q2,
        epochs=50, lr=0.01, save_dir='outputs', opt_name='adagrad'
    )
    plot_history(history_q2)
    ckpt_q2 = torch.load(best_path_q2, map_location=device)
    model_q2.load_state_dict(ckpt_q2['model_state'])
    results_q2 = evaluate_full(model_q2, test_loader_q2, class_names_q2)


### 2-4. Evaluation (30 pts)
- Plot train/val curves.
- Report confusion matrix on test set.
- Discuss most/least correctly classified classes.
- Compare with paper's CNN-MLP: data efficiency?
- Why use separate MLP instead of FC layers at CNN end?
- Role of Global Average Pooling before MLP?
- Why Adagrad for MLP?

In [ ]:
pass


### 2-5-1. Effect of Data Augmentation (10 pts)
- Retrain CNN-MLP without augmentations (only resize + normalize).
- Compare final test accuracy with augmented version.
- Report in table and discuss impact.

### 2-5-2. Role of Optimizer (10 pts)
- Replace Adagrad with Adam; retrain with same settings (including augmentation).
- Compare final test accuracy.
- Discuss: Is Adagrad critical for performance or is Adam comparable/better?

In [ ]:
def fit_with_optimizer(model, train_loader, val_loader, epochs, lr, save_dir, opt_name='adam'):
    criterion = nn.CrossEntropyLoss()
    mlp_params = [p for n, p in model.named_parameters() if 'mlp' in n and p.requires_grad]
    if opt_name.lower() == 'adam':
        optimizer = optim.Adam(mlp_params, lr=lr)
    elif opt_name.lower() == 'adamw':
        optimizer = optim.AdamW(mlp_params, lr=lr, weight_decay=1e-4)
    elif opt_name.lower() == 'adagrad':
        optimizer = optim.Adagrad(mlp_params, lr=lr)
    elif opt_name.lower() == 'sgd':
        optimizer = optim.SGD(mlp_params, lr=lr, momentum=0.9)
    else:
        raise ValueError(f'Unknown optimizer: {opt_name}')
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc, best_path = -1.0, Path(save_dir) / f'best_xception_{opt_name}.pth'
    for e in range(1, epochs + 1):
        t0 = time.time()
        tr_l, tr_a = train_one_epoch(model, train_loader, criterion, optimizer)
        va_l, va_a, _, _ = evaluate(model, val_loader, criterion)
        history['train_loss'].append(tr_l)
        history['train_acc'].append(tr_a)
        history['val_loss'].append(va_l)
        history['val_acc'].append(va_a)
        if va_a > best_acc:
            best_acc = va_a
            torch.save({'model_state': model.state_dict()}, str(best_path))
        print(f"[{opt_name.upper()}] Epoch {e:03d}/{epochs} | train {tr_l:.4f}/{tr_a:.4f} | val {va_l:.4f}/{va_a:.4f} | {time.time()-t0:.1f}s")
    print(f"Best val acc: {best_acc:.4f}")
    return history, best_path


### 2-5-3. Fine-tuning (10 pts)
- Unfreeze top layers of Xception backbone; retrain with smaller LR.
- Compare frozen vs fine-tuned performance.
- Discuss trade-offs: accuracy gain, training time, overfitting risk.


In [ ]:
def unfreeze_top_n_layers(model, n_layers=5):
    if HAS_TIMM:
        all_params = list(model.feature_extractor.named_parameters())
        total = len(all_params)
        for i, (name, param) in enumerate(all_params):
            if i >= total - n_layers:
                param.requires_grad = True
                print(f"Unfroze: {name}")
    else:
        layers = list(model.feature_extractor.children())
        for layer in layers[-n_layers:]:
            for param in layer.parameters():
                param.requires_grad = True

def fit_finetune(model, train_loader, val_loader, epochs, lr_backbone, lr_head, save_dir):
    criterion = nn.CrossEntropyLoss()
    backbone_params = [p for n, p in model.named_parameters() if 'feature_extractor' in n and p.requires_grad]
    mlp_params = [p for n, p in model.named_parameters() if 'mlp' in n and p.requires_grad]
    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': lr_backbone},
        {'params': mlp_params, 'lr': lr_head}
    ], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc, best_path = -1.0, Path(save_dir) / 'best_xception_finetuned.pth'
    for e in range(1, epochs + 1):
        t0 = time.time()
        tr_l, tr_a = train_one_epoch(model, train_loader, criterion, optimizer)
        va_l, va_a, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()
        history['train_loss'].append(tr_l)
        history['train_acc'].append(tr_a)
        history['val_loss'].append(va_l)
        history['val_acc'].append(va_a)
        if va_a > best_acc:
            best_acc = va_a
            torch.save({'model_state': model.state_dict()}, str(best_path))
        print(f"[FINETUNE] Epoch {e:03d}/{epochs} | train {tr_l:.4f}/{tr_a:.4f} | val {va_l:.4f}/{va_a:.4f} | {time.time()-t0:.1f}s")
    print(f"Best val acc: {best_acc:.4f}")
    return history, best_path


In [ ]:
def print_comparison_table():
    print("\n" + "="*80)
    print("Q2 EXPERIMENT SUMMARY")
    print("="*80)
    print(f"{'Experiment':<30} | {'Test Acc':<10} | {'Notes':<30}")
    print("-"*80)
    try:
        print(f"{'Baseline (Xception+MLP, Adagrad)':<30} | {results_q2['test_acc']:.4f}     | With augmentation")
        print(f"{'No Augmentation':<30} | {results_noaug['test_acc']:.4f}     | Frozen backbone")
        print(f"{'Adam Optimizer':<30} | {results_adam['test_acc']:.4f}     | vs Adagrad")
        print(f"{'Fine-tuned (top 10 layers)':<30} | {results_ft['test_acc']:.4f}     | LR_backbone=1e-5")
    except:
        print("Run all Q2 experiments first to populate results.")
    print("="*80)
    print("\nKey Findings:")
    print("- Augmentation impact: [fill in % improvement]")
    print("- Optimizer comparison: [Adagrad vs Adam conclusion]")
    print("- Fine-tuning gain: [frozen vs fine-tuned % difference]")
    print("- Training time trade-off: [fine-tuning takes ~X times longer]")


print_comparison_table()


# Submission Checklist
- [ ] All prompts pasted and answered under each section.
- [ ] Dataset paths configured and splits verified.
- [ ] Model choices justified (baseline vs transfer learning).
- [ ] Hyperparameters listed (epochs, LR, batch size, weight decay).
- [ ] Training curves included.
- [ ] Final test metrics reported (Accuracy, Sensitivity, Specificity).
- [ ] Confusion matrices included and discussed.
- [ ] Error analysis and discussion added.
- [ ] References cited.

In [ ]:
from pprint import pprint

CONFIG = {
    'task': 'rice',  # 'rice' or 'vehicle'
    'data': {
        'rice': {
            'train_dir': '/absolute/path/to/rice/train',
            'val_dir':   '/absolute/path/to/rice/val',
            'test_dir':  '/absolute/path/to/rice/test',
            'num_classes': 4  # Healthy + 3 diseases
        },
        'vehicle': {
            'train_dir': '/absolute/path/to/vehicle/train',
            'val_dir':   '/absolute/path/to/vehicle/val',
            'test_dir':  '/absolute/path/to/vehicle/test',
            'num_classes': 4  # update per dataset
        }
    },
    'img_size': 224,
    'batch_size': 32,
    'epochs': 30,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'num_workers': 2,
    'seed': 42,
    'model': 'alexnet',  # 'alexnet' or 'vgg19_tl'
    'freeze_backbone': True,
    'save_dir': str((Path.cwd() / 'outputs').resolve())
}
Path(CONFIG['save_dir']).mkdir(parents=True, exist_ok=True)
pprint(CONFIG)

In [ ]:
# Transforms
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_transforms(img_size: int = 224, for_transfer: bool = False):
    normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD) if for_transfer else nn.Identity()

    train_tfms = transforms.Compose([
        transforms.Resize(int(img_size * 1.15)),
        transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
        transforms.ToTensor(),
        normalize if for_transfer else transforms.Lambda(lambda x: x)
    ])

    eval_tfms = transforms.Compose([
        transforms.Resize(int(img_size * 1.15)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        normalize if for_transfer else transforms.Lambda(lambda x: x)
    ])
    return train_tfms, eval_tfms

In [ ]:
def build_dataloaders(cfg: dict):
    task = cfg['task']
    data_cfg = cfg['data'][task]
    img_size = cfg['img_size']
    batch_size = cfg['batch_size']
    num_workers = cfg['num_workers']

    train_dir = Path(data_cfg['train_dir'])
    val_dir   = Path(data_cfg['val_dir'])
    test_dir  = Path(data_cfg['test_dir'])

    missing = [p for p in [train_dir, val_dir, test_dir] if not p.exists()]
    if missing:
        print('Missing dataset paths:')
        for m in missing:
            print(' -', m)
        print('Update CONFIG to your local dataset before training.')
        return None

    use_transfer = cfg['model'] in {'vgg19_tl'}
    train_tfms, eval_tfms = get_transforms(img_size, for_transfer=use_transfer)

    train_ds = datasets.ImageFolder(str(train_dir), transform=train_tfms)
    val_ds   = datasets.ImageFolder(str(val_dir),   transform=eval_tfms)
    test_ds  = datasets.ImageFolder(str(test_dir),  transform=eval_tfms)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    class_names = train_ds.classes
    print(f"Classes ({len(class_names)}): {class_names}")
    return train_loader, val_loader, test_loader, class_names

In [ ]:
def show_batch_images(loader, class_names, max_images=8):
    images, labels = next(iter(loader))
    n = min(max_images, images.size(0))
    images = images[:n]
    labels = labels[:n]
    grid = torchvision.utils.make_grid(images, nrow=min(4, n), normalize=True)
    plt.figure(figsize=(8, 6))
    plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
    plt.axis('off')
    plt.title('Sample batch')
    plt.show()

def plot_class_distribution(loader, class_names):
    counts = [0] * len(class_names)
    for _, targets in loader:
        for t in targets.numpy().tolist():
            counts[t] += 1
    plt.figure(figsize=(6, 4))
    sns.barplot(x=class_names, y=counts)
    plt.title('Class counts (by loader batches)')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes: int = 4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
)
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes)
)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [ ]:
def print_model_summary(model, input_size=(3, 224, 224), model_name='Model'):
    print(f"\n{'='*70}")
    print(f"{model_name} Architecture Summary")
    print(f"{'='*70}")
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Total parameters:     {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters:    {total_params - trainable_params:,}")
    print(f"Trainable ratio:      {100 * trainable_params / total_params:.2f}%")
    
    param_size_mb = total_params * 4 / (1024**2) 
    print(f"Estimated size:       {param_size_mb:.2f} MB")
    
    try:
        model.eval()
        with torch.no_grad():
            dummy_input = torch.randn(1, *input_size).to(device)
            output = model(dummy_input)
            print(f"Input shape:          {tuple(dummy_input.shape)}")
            print(f"Output shape:         {tuple(output.shape)}")
    except Exception as e:
        print(f"Could not run forward pass: {e}")
    
    print(f"{'='*70}\n")

try:
    print_model_summary(model_alex, input_size=(3, 224, 224), model_name='AlexNet')
except: pass
try:
    print_model_summary(model_vgg, input_size=(3, 224, 224), model_name='VGG19-TL')
except: pass

try:

    print_model_summary(model_q2, input_size=(3, 299, 299), model_name='Xception-MLP')

except: pass

In [ ]:
def build_vgg19_tl(num_classes: int, freeze_backbone: bool = True):
    try:
        weights = models.VGG19_Weights.IMAGENET1K_V1
    except Exception:
        weights = None
    model = models.vgg19(weights=weights)
    if freeze_backbone:
        for p in model.features.parameters():
            p.requires_grad = False
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model.to(device)

### Helpers — How to Read Results
- Training curves: watch train/val gaps for over/underfitting.
- Test metrics: report overall accuracy and per-class sensitivity/specificity for interpretability.
- Confusion matrix: identify hardest confusions; relate to data (e.g., visually similar classes).
- Keep epochs comparable across models for fair comparisons.

In [ ]:
class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / max(self.count, 1)

def top1_accuracy(logits, targets):
    with torch.no_grad():
        preds = torch.argmax(logits, dim=1)
        correct = (preds == targets).sum().item()
        total = targets.numel()
    return correct / total

def compute_sensitivity_specificity(y_true, y_pred, num_classes):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    sens = []  
    spec = []  
    for c in range(num_classes):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - (TP + FN + FP)
        sens_c = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        spec_c = TN / (TN + FP) if (TN + FP) > 0 else 0.0
        sens.append(sens_c)
        spec.append(spec_c)
    return np.array(sens), np.array(spec), cm

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_meter, acc_meter = AverageMeter(), AverageMeter()
    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        acc = top1_accuracy(outputs, targets)
        loss_meter.update(loss.item(), n=images.size(0))
        acc_meter.update(acc, n=images.size(0))
    return loss_meter.avg, acc_meter.avg

def evaluate(model, loader, criterion):
    model.eval()
    loss_meter, acc_meter = AverageMeter(), AverageMeter()
    all_t, all_p = [], []
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            acc = top1_accuracy(outputs, targets)
            loss_meter.update(loss.item(), n=images.size(0))
            acc_meter.update(acc, n=images.size(0))
            all_t.extend(targets.cpu().numpy().tolist())
            all_p.extend(torch.argmax(outputs, dim=1).cpu().numpy().tolist())
    return loss_meter.avg, acc_meter.avg, np.array(all_t), np.array(all_p)

def fit(model, train_loader, val_loader, epochs, lr, weight_decay, save_dir):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc, best_path = -1.0, Path(save_dir) / 'best_model.pth'

    for e in range(1, epochs + 1):
        t0 = time.time()
        tr_l, tr_a = train_one_epoch(model, train_loader, criterion, optimizer)
        va_l, va_a, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()
        history['train_loss'].append(tr_l)
        history['train_acc'].append(tr_a)
        history['val_loss'].append(va_l)
        history['val_acc'].append(va_a)
        if va_a > best_acc:
            best_acc = va_a
            torch.save({'model_state': model.state_dict()}, str(best_path))
        print(f"Epoch {e:03d}/{epochs} | train {tr_l:.4f}/{tr_a:.4f} | val {va_l:.4f}/{va_a:.4f} | {time.time()-t0:.1f}s")
    print(f"Best val acc: {best_acc:.4f} @ {best_path}")
    return history, best_path

def plot_history(history):
    x = range(1, len(history['train_loss']) + 1)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    axs[0].plot(x, history['train_loss'], label='train')
    axs[0].plot(x, history['val_loss'], label='val')
    axs[0].set_title('Loss')
    axs[0].legend()
    axs[1].plot(x, history['train_acc'], label='train')
    axs[1].plot(x, history['val_acc'], label='val')
    axs[1].set_title('Accuracy')
    axs[1].legend()
    plt.show()

def evaluate_full(model, test_loader, class_names):
    criterion = nn.CrossEntropyLoss()
    tl, ta, y_true, y_pred = evaluate(model, test_loader, criterion)
    sens, spec, cm = compute_sensitivity_specificity(y_true, y_pred, num_classes=len(class_names))
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
    print('Per-class Sensitivity (Recall):', np.round(sens, 4))
    print('Per-class Specificity:', np.round(spec, 4))
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix'); plt.tight_layout(); plt.show()
    return {'test_loss': tl, 'test_acc': ta, 'sens': sens, 'spec': spec, 'cm': cm}

In [ ]:
loaders = build_dataloaders(CONFIG)
if loaders is None:
    print('Configure dataset paths in CONFIG and rerun.')
else:
    train_loader, val_loader, test_loader, class_names = loaders
    num_classes = len(class_names)
    if CONFIG['data'][CONFIG['task']]['num_classes'] != num_classes:
        CONFIG['data'][CONFIG['task']]['num_classes'] = num_classes

    if CONFIG['model'] == 'alexnet':
        model = AlexNet(num_classes=num_classes).to(device)
    elif CONFIG['model'] == 'vgg19_tl':
        model = build_vgg19_tl(num_classes=num_classes, freeze_backbone=CONFIG['freeze_backbone'])
    else:
        raise ValueError('Unknown model in CONFIG["model"]')


    history, best_path = fit(
        model, train_loader, val_loader,
        epochs=CONFIG['epochs'],
        lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'],
        save_dir=CONFIG['save_dir']
    )
    plot_history(history)

    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    _ = evaluate_full(model, test_loader, class_names)